# Task 2 — Unsupervised Domain Adaptation (PACS)
Restructured for idempotent re-runs: every section is self-contained top to bottom.
**Run order matters** — each section below depends only on sections above it, never on
anything from a previous Kaggle session's memory. If your session restarts, just
Run All from the top; nothing here assumes leftover state.


## 0. Setup — imports, seed, BatchNorm-freeze utility
Run this first, always. `set_seed` is called once here; every training cell below
re-calls it before building its model so results stay reproducible under seed 6304
regardless of what ran earlier in the session.


In [1]:
import os
import copy
import random
import numpy as np

import torch
import torch.nn as nn
from torch.autograd import Function
from torch.optim import AdamW
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

def set_seed(seed=6304):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(6304)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def freeze_bn_stats(m):
    """Freezes BatchNorm running statistics at their pretrained values but
    keeps scale (gamma) and bias (beta) trainable. Call model.apply(freeze_bn_stats)
    AFTER model.train() every epoch, for every method in Tasks 2 and 3."""
    if m.__class__.__name__.find('BatchNorm') != -1:
        m.eval()

print("Setup complete.")


Using device: cuda
Setup complete.


## 1. Data — PACS loaders (Photo / Art Painting / Cartoon source, Sketch target)
Stratified 80/20 source splits with seed 6304, per spec. Source train batches give
8 examples/domain (24 total); target batches give 24 examples, matching total source
and target batch sizes as required.


In [2]:
# Auto-locate the dataset root in Kaggle's mounted input folder
base_dir = ""
for root, dirs, files in os.walk('/kaggle/input/'):
    if {'photo', 'art_painting', 'cartoon', 'sketch'}.issubset({d.lower() for d in dirs}):
        base_dir = root
        break

print(f"Dataset located at: {base_dir}")

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_stratified_split(dataset, seed=6304, test_size=0.2):
    labels = dataset.targets
    train_idx, val_idx = train_test_split(
        range(len(labels)), test_size=test_size, stratify=labels, random_state=seed
    )
    return train_idx, val_idx

# Source loaders: Photo, Art Painting, Cartoon
sources = ['photo', 'art_painting', 'cartoon']
source_loaders = {}

for domain in sources:
    domain_path = os.path.join(base_dir, domain)
    full_train_ds = ImageFolder(root=domain_path, transform=train_transform)
    full_val_ds = ImageFolder(root=domain_path, transform=eval_transform)

    train_idx, val_idx = get_stratified_split(full_train_ds, seed=6304)

    train_subset = Subset(full_train_ds, train_idx)
    val_subset = Subset(full_val_ds, val_idx)

    source_loaders[domain] = {
        'train': DataLoader(train_subset, batch_size=8, shuffle=True, drop_last=True),
        'val': DataLoader(val_subset, batch_size=32, shuffle=False)
    }

# Target loader: Sketch
target_path = os.path.join(base_dir, 'sketch')
target_ds_train = ImageFolder(root=target_path, transform=train_transform)
target_ds_eval = ImageFolder(root=target_path, transform=eval_transform)

target_loader_train = DataLoader(target_ds_train, batch_size=24, shuffle=True, drop_last=True)
target_loader_eval = DataLoader(target_ds_eval, batch_size=32, shuffle=False)

print("Data loaders ready:")
for d in sources:
    print(f"  {d}: {len(source_loaders[d]['train'].dataset)} train / {len(source_loaders[d]['val'].dataset)} val")
print(f"  sketch (target): {len(target_ds_train)} images")


Dataset located at: /kaggle/input/datasets/nickfratto/pacs-dataset/pacs_data/pacs_data
Data loaders ready:
  photo: 1336 train / 334 val
  art_painting: 1638 train / 410 val
  cartoon: 1875 train / 469 val
  sketch (target): 3929 images


## 2. Shared evaluation utility
Works for ERM, DANN, and CDAN alike — every model here returns plain class logits
from `forward(x)` when called without `alpha`.


In [3]:
def evaluate_loader(model, loader):
    """Computes Top-1 Accuracy and Macro-F1 for a given dataloader."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return acc * 100, macro_f1 * 100

def report_domains(model, tag):
    print(f"--- {tag} Evaluation ---")
    for domain in ['photo', 'art_painting', 'cartoon']:
        acc, f1 = evaluate_loader(model, source_loaders[domain]['val'])
        print(f"Source Val [{domain.capitalize()}]: Accuracy = {acc:.2f}% | Macro-F1 = {f1:.2f}%")
    target_acc, target_f1 = evaluate_loader(model, target_loader_eval)
    print(f"Target [Sketch]: Accuracy = {target_acc:.2f}% | Macro-F1 = {target_f1:.2f}%")
    return target_acc, target_f1


## 3. Source-Only ERM baseline
This checkpoint is also reused unchanged as the Task 3 ERM baseline — don't retrain
it differently there.


In [4]:
# set_seed(6304)

# erm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
# erm_model.fc = nn.Linear(erm_model.fc.in_features, 7)
# erm_model.to(device)

# optimizer = AdamW(erm_model.parameters(), lr=1e-4, weight_decay=1e-4)
# criterion = nn.CrossEntropyLoss()

# max_epochs = 30
# patience = 5
# best_mean_f1 = -1
# epochs_no_improve = 0
# best_state = None

# photo_loader = source_loaders['photo']['train']
# art_loader = source_loaders['art_painting']['train']
# cartoon_loader = source_loaders['cartoon']['train']

# print("Starting Source-Only ERM Training...")

# for epoch in range(max_epochs):
#     erm_model.train()
#     erm_model.apply(freeze_bn_stats)

#     total_loss = 0
#     for (img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c) in zip(photo_loader, art_loader, cartoon_loader):
#         images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
#         labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)

#         optimizer.zero_grad()
#         outputs = erm_model(images)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     erm_model.eval()
#     f1s = [evaluate_loader(erm_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
#     mean_f1 = np.mean(f1s)
#     print(f"Epoch {epoch+1}/{max_epochs} | Loss: {total_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

#     if mean_f1 > best_mean_f1:
#         best_mean_f1 = mean_f1
#         best_state = copy.deepcopy(erm_model.state_dict())
#         epochs_no_improve = 0
#     else:
#         epochs_no_improve += 1
#         if epochs_no_improve >= patience:
#             print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
#             break

# torch.save(best_state, '/kaggle/working/source_only_erm.pth')
# print(f"ERM training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved.")


In [5]:
# ERM — load only, don't retrain
erm_model = models.resnet18(weights=None)
erm_model.fc = nn.Linear(erm_model.fc.in_features, 7)
erm_model.load_state_dict(torch.load('/kaggle/working/source_only_erm.pth'))
erm_model.to(device)

print("ERM checkpoint loaded from disk.")

ERM checkpoint loaded from disk.


In [6]:
# --- ERM evaluation ---
erm_model.load_state_dict(torch.load('/kaggle/working/source_only_erm.pth'))
erm_model.to(device)
erm_target_acc, erm_target_f1 = report_domains(erm_model, "Source-Only ERM")


--- Source-Only ERM Evaluation ---
Source Val [Photo]: Accuracy = 96.41% | Macro-F1 = 95.82%
Source Val [Art_painting]: Accuracy = 92.20% | Macro-F1 = 91.72%
Source Val [Cartoon]: Accuracy = 94.46% | Macro-F1 = 94.43%
Target [Sketch]: Accuracy = 72.46% | Macro-F1 = 74.41%


## 3b. DAN — MMD Alignment
### Architecture + MMD utility
No adversarial component here — no GRL, no discriminator. The backbone just extracts
features, a linear head classifies them, and a multi-kernel MMD penalty is added
directly to the classification loss during training (defined below). Because there's
no reversed gradient, this should train far more stably than DANN/CDAN did.

In [17]:
class DAN_ResNet(nn.Module):
    def __init__(self, num_classes=7):
        super(DAN_ResNet, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        in_features = resnet.fc.in_features
        self.class_classifier = nn.Linear(in_features, num_classes)

    def forward(self, x, return_features=False):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        class_preds = self.class_classifier(features)
        if return_features:
            return class_preds, features
        return class_preds


def compute_mmd(source_feats, target_feats):
    """Multi-kernel MMD^2 between two feature sets, using a sum of three RBF
    kernels with bandwidths 0.5x, 1x, 2x the median pairwise squared distance
    of the combined batch, per spec."""
    combined = torch.cat([source_feats, target_feats], dim=0)
    n = combined.size(0)
    dist_sq = torch.cdist(combined, combined, p=2) ** 2

    off_diag_mask = ~torch.eye(n, dtype=torch.bool, device=combined.device)
    median_sq_dist = dist_sq[off_diag_mask].median()
    if median_sq_dist <= 0:
        median_sq_dist = torch.tensor(1.0, device=combined.device)

    bandwidths = [0.5 * median_sq_dist, 1.0 * median_sq_dist, 2.0 * median_sq_dist]
    K = sum(torch.exp(-dist_sq / bw) for bw in bandwidths)

    n_s = source_feats.size(0)
    K_ss = K[:n_s, :n_s]
    K_tt = K[n_s:, n_s:]
    K_st = K[:n_s, n_s:]

    mmd2 = K_ss.mean() + K_tt.mean() - 2 * K_st.mean()
    return mmd2

print("DAN architecture and MMD utility defined.")

DAN architecture and MMD utility defined.


### Training
`lambda_mmd = 1` for the main comparison, per spec. Same AdamW settings, seed reset,
grad clipping, checkpoint selection, and early stopping pattern as the other methods
— kept for consistency even though DAN shouldn't need the stability safety nets as
much as DANN/CDAN did.

In [18]:
set_seed(6304)

dan_model = DAN_ResNet()
dan_model.to(device)

optimizer = AdamW(dan_model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
class_criterion = nn.CrossEntropyLoss()
lambda_mmd = 1.0

max_epochs = 30
patience = 5
best_mean_f1 = -1
epochs_no_improve = 0
best_state = None

photo_loader = source_loaders['photo']['train']
art_loader = source_loaders['art_painting']['train']
cartoon_loader = source_loaders['cartoon']['train']

print("Starting DAN (MMD Alignment) Training...")

for epoch in range(max_epochs):
    dan_model.train()
    dan_model.apply(freeze_bn_stats)

    target_iter = iter(target_loader_train)
    total_class_loss, total_mmd_loss = 0, 0

    for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
        zip(photo_loader, art_loader, cartoon_loader)):

        try:
            img_t, _ = next(target_iter)
        except StopIteration:
            target_iter = iter(target_loader_train)
            img_t, _ = next(target_iter)

        src_images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
        src_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)
        tgt_images = img_t.to(device)

        src_class_preds, src_features = dan_model(src_images, return_features=True)
        _, tgt_features = dan_model(tgt_images, return_features=True)

        loss_class = class_criterion(src_class_preds, src_labels)
        loss_mmd = compute_mmd(src_features, tgt_features)

        total_loss = loss_class + lambda_mmd * loss_mmd

        optimizer.zero_grad()
        total_loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(dan_model.parameters(), max_norm=1.0)

        if not torch.isfinite(grad_norm):
            optimizer.zero_grad()
            if batch_idx % 20 == 0:
                print(f"  batch {batch_idx}: SKIPPED (non-finite grad)")
            continue

        optimizer.step()

        total_class_loss += loss_class.item()
        total_mmd_loss += loss_mmd.item()

        if batch_idx % 20 == 0:
            print(f"  batch {batch_idx}: class={loss_class.item():.3f}, "
                  f"mmd={loss_mmd.item():.4f}, grad_norm={grad_norm.item():.3f}")

    dan_model.eval()
    f1s = [evaluate_loader(dan_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
    mean_f1 = np.mean(f1s)

    print(f"Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
          f"MMD Loss: {total_mmd_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        best_state = copy.deepcopy(dan_model.state_dict())
        epochs_no_improve = 0
        torch.save(best_state, '/kaggle/working/dan_model.pth')
        print(f"  → new best (F1={mean_f1:.2f}%), checkpoint saved.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
            break

print(f"DAN training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved to /kaggle/working/dan_model.pth.")

Starting DAN (MMD Alignment) Training...
  batch 0: class=2.077, mmd=1.0048, grad_norm=16.892
  batch 20: class=0.752, mmd=0.2517, grad_norm=13.060
  batch 40: class=0.905, mmd=0.2307, grad_norm=18.636
  batch 60: class=0.585, mmd=0.2366, grad_norm=12.105
  batch 80: class=0.207, mmd=0.3209, grad_norm=11.549
  batch 100: class=0.352, mmd=0.2771, grad_norm=12.812
  batch 120: class=0.123, mmd=0.2005, grad_norm=7.440
  batch 140: class=0.298, mmd=0.2730, grad_norm=9.812
  batch 160: class=0.129, mmd=0.2160, grad_norm=9.139
Epoch 1/30 | Class Loss: 92.9799 | MMD Loss: 38.8146 | Mean Source Val F1: 89.09%
  → new best (F1=89.09%), checkpoint saved.
  batch 0: class=0.273, mmd=0.1666, grad_norm=13.512
  batch 20: class=0.254, mmd=0.1976, grad_norm=11.365
  batch 40: class=0.212, mmd=0.2292, grad_norm=15.029
  batch 60: class=0.268, mmd=0.1740, grad_norm=10.140
  batch 80: class=0.205, mmd=0.1326, grad_norm=14.902
  batch 100: class=0.171, mmd=0.1876, grad_norm=10.398
  batch 120: class=0.18

In [19]:
# --- DAN evaluation ---
dan_model.load_state_dict(torch.load('/kaggle/working/dan_model.pth'))
dan_model.to(device)
dan_target_acc, dan_target_f1 = report_domains(dan_model, "DAN")

--- DAN Evaluation ---
Source Val [Photo]: Accuracy = 95.81% | Macro-F1 = 95.01%
Source Val [Art_painting]: Accuracy = 87.56% | Macro-F1 = 87.18%
Source Val [Cartoon]: Accuracy = 92.54% | Macro-F1 = 93.40%
Target [Sketch]: Accuracy = 73.58% | Macro-F1 = 71.76%


In [ ]:
torch.save(dan_model.state_dict(), '/kaggle/working/dan_model.pth')
torch.save(cdan_model.state_dict(), '/kaggle/working/cdan_model.pth')
print("Re-saved dan_model.pth and cdan_model.pth")

## 4. DANN — Adversarial Alignment
### 4a. Architecture (Gradient Reversal Layer + model)


In [7]:
class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None


class DANN_ResNet(nn.Module):
    def __init__(self, num_classes=7):
        super(DANN_ResNet, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        in_features = resnet.fc.in_features

        self.class_classifier = nn.Linear(in_features, num_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )

    def forward(self, x, alpha=None):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        class_preds = self.class_classifier(features)

        if alpha is not None:
            reversed_features = GradientReversalFn.apply(features, alpha)
            domain_preds = self.domain_classifier(reversed_features)
            return class_preds, domain_preds

        return class_preds

print("DANN architecture defined.")


DANN architecture defined.


### 4b. Training
AdamW with `eps=1e-4` (loosened from default `1e-8` for GRL stability), grad-norm
clipping, seed reset, per-batch diagnostic logging, and checkpoint selection with
early stopping on mean source-validation macro-F1.


In [8]:
# set_seed(6304)

# dann_model = DANN_ResNet()
# dann_model.to(device)

# optimizer = AdamW(dann_model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
# class_criterion = nn.CrossEntropyLoss()
# domain_criterion = nn.CrossEntropyLoss()

# max_epochs = 30
# patience = 5
# best_mean_f1 = -1
# epochs_no_improve = 0
# best_state = None

# photo_loader = source_loaders['photo']['train']
# art_loader = source_loaders['art_painting']['train']
# cartoon_loader = source_loaders['cartoon']['train']
# num_batches = min(len(photo_loader), len(art_loader), len(cartoon_loader))

# print("Starting DANN Adversarial Training...")

# for epoch in range(max_epochs):
#     dann_model.train()
#     dann_model.apply(freeze_bn_stats)

#     target_iter = iter(target_loader_train)
#     total_class_loss, total_domain_loss = 0, 0

#     for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
#         zip(photo_loader, art_loader, cartoon_loader)):

#         try:
#             img_t, _ = next(target_iter)
#         except StopIteration:
#             target_iter = iter(target_loader_train)
#             img_t, _ = next(target_iter)

#         p = float(batch_idx + epoch * num_batches) / (max_epochs * num_batches)
#         alpha = 2. / (1. + np.exp(-10 * p)) - 1

#         src_images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
#         src_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)
#         src_domain_labels = torch.zeros(src_images.size(0), dtype=torch.long).to(device)

#         src_class_preds, src_domain_preds = dann_model(src_images, alpha=alpha)
#         loss_class = class_criterion(src_class_preds, src_labels)
#         loss_domain_src = domain_criterion(src_domain_preds, src_domain_labels)

#         tgt_images = img_t.to(device)
#         tgt_domain_labels = torch.ones(tgt_images.size(0), dtype=torch.long).to(device)
#         _, tgt_domain_preds = dann_model(tgt_images, alpha=alpha)
#         loss_domain_tgt = domain_criterion(tgt_domain_preds, tgt_domain_labels)

#         loss_domain = loss_domain_src + loss_domain_tgt
#         total_loss = loss_class + loss_domain

#         optimizer.zero_grad()
#         total_loss.backward()

#         grad_norm = torch.nn.utils.clip_grad_norm_(dann_model.parameters(), max_norm=1.0)

#         if not torch.isfinite(grad_norm):
#             optimizer.zero_grad()
#             if batch_idx % 20 == 0:
#                 print(f"  batch {batch_idx}: SKIPPED (non-finite grad)")
#             continue

#         optimizer.step()

#         total_class_loss += loss_class.item()
#         total_domain_loss += loss_domain.item()

#         if batch_idx % 20 == 0:
#             print(f"  batch {batch_idx}: class={loss_class.item():.3f}, "
#                   f"domain={loss_domain.item():.3f}, grad_norm={grad_norm.item():.3f}, alpha={alpha:.3f}")

#     dann_model.eval()
#     f1s = [evaluate_loader(dann_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
#     mean_f1 = np.mean(f1s)

#     print(f"Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
#           f"Domain Loss: {total_domain_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

#     if mean_f1 > best_mean_f1:
#         best_mean_f1 = mean_f1
#         best_state = copy.deepcopy(dann_model.state_dict())
#         epochs_no_improve = 0
#     else:
#         epochs_no_improve += 1
#         if epochs_no_improve >= patience:
#             print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
#             break

# torch.save(best_state, '/kaggle/working/dann_model.pth')
# print(f"DANN training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved.")


In [9]:
# DANN — load only, don't retrain
dann_model = DANN_ResNet()
dann_model.load_state_dict(torch.load('/kaggle/working/dann_model.pth'))
dann_model.to(device)

print("DANN checkpoint loaded from disk.")

DANN checkpoint loaded from disk.


In [10]:
# --- DANN evaluation ---
dann_model.load_state_dict(torch.load('/kaggle/working/dann_model.pth'))
dann_model.to(device)
dann_target_acc, dann_target_f1 = report_domains(dann_model, "DANN")


--- DANN Evaluation ---
Source Val [Photo]: Accuracy = 79.64% | Macro-F1 = 73.59%
Source Val [Art_painting]: Accuracy = 66.10% | Macro-F1 = 61.09%
Source Val [Cartoon]: Accuracy = 56.93% | Macro-F1 = 55.65%
Target [Sketch]: Accuracy = 30.54% | Macro-F1 = 14.34%


## 5. CDAN — Class-Conditional Adversarial Alignment
### 5a. Architecture
Same GRL, same discriminator hidden width/dropout/schedule as DANN. The only
structural difference: the discriminator sees `g(x) = vec(f ⊗ p)` instead of
raw features `f`. No detaching `f` or `p`, per spec.


In [11]:
class CDAN_ResNet(nn.Module):
    def __init__(self, num_classes=7):
        super(CDAN_ResNet, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        in_features = resnet.fc.in_features
        self.num_classes = num_classes

        self.class_classifier = nn.Linear(in_features, num_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(in_features * num_classes, 256),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )

    def forward(self, x, alpha=None):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        class_preds = self.class_classifier(features)

        if alpha is not None:
            probs = torch.softmax(class_preds, dim=1)
            g = torch.bmm(probs.unsqueeze(2), features.unsqueeze(1))  # (batch, num_classes, in_features)
            g = g.view(g.size(0), -1)
            reversed_g = GradientReversalFn.apply(g, alpha)
            domain_preds = self.domain_classifier(reversed_g)
            return class_preds, domain_preds

        return class_preds

print("CDAN architecture defined.")


CDAN architecture defined.


### 5b. Training


In [12]:
set_seed(6304)

cdan_model = CDAN_ResNet()
cdan_model.to(device)

optimizer = AdamW(cdan_model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
class_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.CrossEntropyLoss()

max_epochs = 30
patience = 5
best_mean_f1 = -1
epochs_no_improve = 0
best_state = None

photo_loader = source_loaders['photo']['train']
art_loader = source_loaders['art_painting']['train']
cartoon_loader = source_loaders['cartoon']['train']
num_batches = min(len(photo_loader), len(art_loader), len(cartoon_loader))

print("Starting CDAN Adversarial Training...")

for epoch in range(max_epochs):
    cdan_model.train()
    cdan_model.apply(freeze_bn_stats)

    target_iter = iter(target_loader_train)
    total_class_loss, total_domain_loss = 0, 0

    for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
        zip(photo_loader, art_loader, cartoon_loader)):

        try:
            img_t, _ = next(target_iter)
        except StopIteration:
            target_iter = iter(target_loader_train)
            img_t, _ = next(target_iter)

        p = float(batch_idx + epoch * num_batches) / (max_epochs * num_batches)
        alpha = 2. / (1. + np.exp(-10 * p)) - 1

        src_images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
        src_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)
        src_domain_labels = torch.zeros(src_images.size(0), dtype=torch.long).to(device)

        src_class_preds, src_domain_preds = cdan_model(src_images, alpha=alpha)
        loss_class = class_criterion(src_class_preds, src_labels)
        loss_domain_src = domain_criterion(src_domain_preds, src_domain_labels)

        tgt_images = img_t.to(device)
        tgt_domain_labels = torch.ones(tgt_images.size(0), dtype=torch.long).to(device)
        _, tgt_domain_preds = cdan_model(tgt_images, alpha=alpha)
        loss_domain_tgt = domain_criterion(tgt_domain_preds, tgt_domain_labels)

        loss_domain = loss_domain_src + loss_domain_tgt
        total_loss = loss_class + loss_domain

        optimizer.zero_grad()
        total_loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(cdan_model.parameters(), max_norm=1.0)

        if not torch.isfinite(grad_norm):
            optimizer.zero_grad()
            if batch_idx % 20 == 0:
                print(f"  batch {batch_idx}: SKIPPED (non-finite grad)")
            continue

        optimizer.step()

        total_class_loss += loss_class.item()
        total_domain_loss += loss_domain.item()

        if batch_idx % 20 == 0:
            print(f"  batch {batch_idx}: class={loss_class.item():.3f}, "
                  f"domain={loss_domain.item():.3f}, grad_norm={grad_norm.item():.3f}, alpha={alpha:.3f}")

    cdan_model.eval()
    f1s = [evaluate_loader(cdan_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
    mean_f1 = np.mean(f1s)

    print(f"Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
          f"Domain Loss: {total_domain_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        best_state = copy.deepcopy(cdan_model.state_dict())
        epochs_no_improve = 0
        torch.save(best_state, '/kaggle/working/cdan_model.pth')  # write immediately, don't wait for the loop to end
        print(f"  → new best (F1={mean_f1:.2f}%), checkpoint saved.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
            break

print(f"CDAN training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved to /kaggle/working/cdan_model.pth.")

Starting CDAN Adversarial Training...
  batch 0: class=2.321, domain=1.413, grad_norm=13.803, alpha=0.000
  batch 20: class=0.707, domain=0.932, grad_norm=18.571, alpha=0.020
  batch 40: class=0.447, domain=0.663, grad_norm=12.640, alpha=0.040
  batch 60: class=0.501, domain=0.580, grad_norm=13.227, alpha=0.060
  batch 80: class=0.527, domain=0.880, grad_norm=13.057, alpha=0.080
  batch 100: class=0.824, domain=1.024, grad_norm=28.328, alpha=0.099
  batch 120: class=0.144, domain=1.079, grad_norm=15.692, alpha=0.119
  batch 140: class=1.120, domain=1.654, grad_norm=37.626, alpha=0.139
  batch 160: class=0.586, domain=2.764, grad_norm=45.835, alpha=0.158
Epoch 1/30 | Class Loss: 104.9231 | Domain Loss: 250.0542 | Mean Source Val F1: 83.56%
  → new best (F1=83.56%), checkpoint saved.
  batch 0: class=0.238, domain=2.351, grad_norm=29.834, alpha=0.165
  batch 20: class=0.454, domain=1.675, grad_norm=30.051, alpha=0.184
  batch 40: class=0.354, domain=2.035, grad_norm=33.021, alpha=0.204
 

In [13]:
# --- CDAN evaluation ---
cdan_model.load_state_dict(torch.load('/kaggle/working/cdan_model.pth'))
cdan_model.to(device)
cdan_target_acc, cdan_target_f1 = report_domains(cdan_model, "CDAN")


--- CDAN Evaluation ---
Source Val [Photo]: Accuracy = 96.71% | Macro-F1 = 96.23%
Source Val [Art_painting]: Accuracy = 90.73% | Macro-F1 = 90.91%
Source Val [Cartoon]: Accuracy = 92.54% | Macro-F1 = 93.40%
Target [Sketch]: Accuracy = 35.71% | Macro-F1 = 37.05%


## 6. Domain Separability Diagnostic
For each frozen backbone (ERM, DAN, DANN, CDAN): collect equal numbers of
source-validation and target (Sketch) features, split 70/30 with seed 6304, and
train a balanced logistic-regression classifier (C=1) to distinguish source from
target. Its held-out accuracy is the domain-separability score — 50% is chance
(domains indistinguishable), 100% means fully separable.


In [26]:
from torch.utils.data import ConcatDataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler  # added

def get_feature_extractor(model):
    """DAN/DANN/CDAN expose .feature_extractor directly; the plain ERM ResNet
    doesn't, so build an equivalent extractor from its children (everything up
    to and including avgpool, excluding the final fc layer)."""
    if hasattr(model, 'feature_extractor'):
        return model.feature_extractor
    return nn.Sequential(*list(model.children())[:-1])

def extract_features(model, loader):
    model.eval()
    extractor = get_feature_extractor(model)
    feats = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            f = extractor(images)
            f = f.view(f.size(0), -1)
            feats.append(f.cpu())
    return torch.cat(feats, dim=0).numpy()

# Combined source-validation set across all three source domains
source_val_combined = ConcatDataset([source_loaders[d]['val'].dataset for d in ['photo', 'art_painting', 'cartoon']])
source_val_loader = DataLoader(source_val_combined, batch_size=32, shuffle=False)

def domain_separability(model, tag):
    src_feats = extract_features(model, source_val_loader)
    tgt_feats = extract_features(model, target_loader_eval)

    # Equal numbers of source and target features, sampled deterministically
    rng = np.random.RandomState(6304)
    n = min(len(src_feats), len(tgt_feats))
    src_idx = rng.choice(len(src_feats), size=n, replace=False)
    tgt_idx = rng.choice(len(tgt_feats), size=n, replace=False)
    src_feats, tgt_feats = src_feats[src_idx], tgt_feats[tgt_idx]

    X = np.concatenate([src_feats, tgt_feats], axis=0)
    y = np.concatenate([np.zeros(n), np.ones(n)])  # 0 = source, 1 = target

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.3, random_state=6304, stratify=y
    )
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    clf = LogisticRegression(C=1, class_weight='balanced', max_iter=5000, random_state=6304)
    clf.fit(X_train, y_train)
    separability = clf.score(X_val, y_val) * 100

    print(f"{tag:<8} domain separability: {separability:.2f}%  (n={n} per domain, chance=50%)")
    return separability

print("--- Domain Separability Diagnostic ---")
erm_sep = domain_separability(erm_model, "ERM")
dan_sep = domain_separability(dan_model, "DAN")
dann_sep = domain_separability(dann_model, "DANN")
cdan_sep = domain_separability(cdan_model, "CDAN")

--- Domain Separability Diagnostic ---
ERM      domain separability: 99.73%  (n=1213 per domain, chance=50%)
DAN      domain separability: 92.58%  (n=1213 per domain, chance=50%)
DANN     domain separability: 100.00%  (n=1213 per domain, chance=50%)
CDAN     domain separability: 98.35%  (n=1213 per domain, chance=50%)


In [27]:
dan_sweep_results[0.1]['separability'] = domain_separability(dan_sweep_models[0.1], "DAN(lambda=0.1)")
dan_sweep_results[10]['separability'] = domain_separability(dan_sweep_models[10], "DAN(lambda=10)")
dan_sweep_results[1]['separability'] = dan_sep

print("\n--- DAN Controlled Design Study: lambda_mmd sweep (updated separability) ---")
print(f"{'Lambda':<10}{'Source F1':>12}{'Sketch Acc':>12}{'Sketch F1':>12}{'Separability':>14}")
for lam in [0.1, 1, 10]:
    r = dan_sweep_results[lam]
    print(f"{lam:<10}{r['source_f1']:>11.2f}%{r['target_acc']:>11.2f}%{r['target_f1']:>11.2f}%{r['separability']:>13.2f}%")

DAN(lambda=0.1) domain separability: 95.74%  (n=1213 per domain, chance=50%)
DAN(lambda=10) domain separability: 94.09%  (n=1213 per domain, chance=50%)

--- DAN Controlled Design Study: lambda_mmd sweep (updated separability) ---
Lambda       Source F1  Sketch Acc   Sketch F1  Separability
0.1             95.09%      68.52%      63.59%        95.74%
1               91.86%      73.58%      71.76%        92.58%
10              89.11%      69.25%      65.94%        94.09%


## 7. Per-Class Accuracy Analysis
Per-class Sketch accuracy for each method against the ERM baseline, plus dominant
confusions for the two methods that showed negative transfer (DANN, CDAN).

In [23]:
from sklearn.metrics import confusion_matrix

class_names = target_ds_eval.classes  # PACS class names, in ImageFolder's index order
print("Class index mapping:", {i: name for i, name in enumerate(class_names)})

def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

def per_class_accuracy(labels, preds, num_classes):
    cm = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    with np.errstate(divide='ignore', invalid='ignore'):
        acc = np.diag(cm) / cm.sum(axis=1)
    return acc * 100, cm

models_dict = {'ERM': erm_model, 'DAN': dan_model, 'DANN': dann_model, 'CDAN': cdan_model}

results = {}
confmats = {}
for name, model in models_dict.items():
    labels, preds = get_predictions(model, target_loader_eval)
    acc, cm = per_class_accuracy(labels, preds, num_classes=len(class_names))
    results[name] = acc
    confmats[name] = cm

# --- per-class accuracy table across all four methods ---
print("\n--- Per-Class Sketch Accuracy ---")
header = f"{'Class':<12}" + "".join(f"{m:>10}" for m in models_dict)
print(header)
for i, cname in enumerate(class_names):
    row = f"{cname:<12}" + "".join(f"{results[m][i]:>9.1f}%" for m in models_dict)
    print(row)

# --- change relative to ERM, sorted worst to best ---
for name in ['DAN', 'DANN', 'CDAN']:
    deltas = results[name] - results['ERM']
    order = np.argsort(deltas)
    print(f"\n--- {name} vs ERM (sorted worst → best) ---")
    for idx in order:
        cname = class_names[idx]
        print(f"  {cname:<12} {deltas[idx]:+6.1f} pts  (ERM {results['ERM'][idx]:.1f}% -> {name} {results[name][idx]:.1f}%)")

Class index mapping: {0: 'dog', 1: 'elephant', 2: 'giraffe', 3: 'guitar', 4: 'horse', 5: 'house', 6: 'person'}

--- Per-Class Sketch Accuracy ---
Class              ERM       DAN      DANN      CDAN
dog              54.9%     51.2%     93.0%      5.1%
elephant         96.1%     93.4%      0.0%      1.2%
giraffe          67.6%     63.3%      0.0%     54.3%
guitar           93.9%     89.8%     79.3%     72.2%
horse            58.0%     67.3%      0.0%     37.1%
house            70.0%     98.8%      0.0%    100.0%
person           64.4%     96.2%      0.0%     77.5%

--- DAN vs ERM (sorted worst → best) ---
  giraffe        -4.2 pts  (ERM 67.6% -> DAN 63.3%)
  guitar         -4.1 pts  (ERM 93.9% -> DAN 89.8%)
  dog            -3.8 pts  (ERM 54.9% -> DAN 51.2%)
  elephant       -2.7 pts  (ERM 96.1% -> DAN 93.4%)
  horse          +9.3 pts  (ERM 58.0% -> DAN 67.3%)
  house         +28.8 pts  (ERM 70.0% -> DAN 98.8%)
  person        +31.9 pts  (ERM 64.4% -> DAN 96.2%)

--- DANN vs ERM (sorted

In [24]:
def top_confusion(cm, class_names, true_idx):
    row = cm[true_idx].copy()
    row[true_idx] = 0  # exclude correct predictions
    total = cm[true_idx].sum()
    if row.sum() == 0:
        return None, 0, total
    pred_idx = np.argmax(row)
    return class_names[pred_idx], row[pred_idx], total

print("--- Dominant confusions: 3 worst-hit classes per method (vs ERM) ---")
for name in ['DANN', 'CDAN']:
    deltas = results[name] - results['ERM']
    worst_classes = np.argsort(deltas)[:3]
    print(f"\n{name}:")
    for idx in worst_classes:
        cname = class_names[idx]
        pred_name, count, total = top_confusion(confmats[name], class_names, idx)
        print(f"  {cname}: {results[name][idx]:.1f}% acc — most often predicted as "
              f"'{pred_name}' ({count}/{total} examples)")

--- Dominant confusions: 3 worst-hit classes per method (vs ERM) ---

DANN:
  elephant: 0.0% acc — most often predicted as 'dog' (715/740 examples)
  house: 0.0% acc — most often predicted as 'dog' (51/80 examples)
  giraffe: 0.0% acc — most often predicted as 'dog' (536/753 examples)

CDAN:
  elephant: 1.2% acc — most often predicted as 'person' (656/740 examples)
  dog: 5.1% acc — most often predicted as 'person' (662/772 examples)
  guitar: 72.2% acc — most often predicted as 'person' (85/608 examples)


## 8. Controlled Design Study — DAN λ_MMD Sweep
Hypothesis: increasing λ_MMD should push the backbone to weight the MMD penalty more
heavily relative to the classification loss, so we'd expect domain separability to fall
(features pushed closer together) at the cost of some source accuracy, with an uncertain
effect on target recognition — since Task 2's diagnostic already showed separability and
target accuracy are not tightly coupled for the adversarial methods, we don't assume
lower separability here will straightforwardly help Sketch either.
λ=1 reuses the existing DAN run above rather than retraining it.

In [25]:
lambda_values_to_train = [0.1, 10]
dan_sweep_models = {}
dan_sweep_results = {
    1: {'source_f1': best_mean_f1 if 'dan_model' in dir() else None,  # falls back below if unavailable
        'target_acc': dan_target_acc, 'target_f1': dan_target_f1, 'separability': dan_sep}
}

for lam in lambda_values_to_train:
    print(f"\n=== Training DAN with lambda_mmd={lam} ===")
    set_seed(6304)
    model = DAN_ResNet()
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
    class_criterion = nn.CrossEntropyLoss()

    max_epochs = 30
    patience = 5
    best_mean_f1 = -1
    epochs_no_improve = 0
    best_state = None

    photo_loader = source_loaders['photo']['train']
    art_loader = source_loaders['art_painting']['train']
    cartoon_loader = source_loaders['cartoon']['train']

    for epoch in range(max_epochs):
        model.train()
        model.apply(freeze_bn_stats)
        target_iter = iter(target_loader_train)
        total_class_loss, total_mmd_loss = 0, 0

        for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
            zip(photo_loader, art_loader, cartoon_loader)):

            try:
                img_t, _ = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader_train)
                img_t, _ = next(target_iter)

            src_images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
            src_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)
            tgt_images = img_t.to(device)

            src_class_preds, src_features = model(src_images, return_features=True)
            _, tgt_features = model(tgt_images, return_features=True)

            loss_class = class_criterion(src_class_preds, src_labels)
            loss_mmd = compute_mmd(src_features, tgt_features)
            total_loss = loss_class + lam * loss_mmd

            optimizer.zero_grad()
            total_loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                optimizer.zero_grad()
                continue
            optimizer.step()

            total_class_loss += loss_class.item()
            total_mmd_loss += loss_mmd.item()

        model.eval()
        f1s = [evaluate_loader(model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
        mean_f1 = np.mean(f1s)
        print(f"  Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
              f"MMD Loss: {total_mmd_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(best_state, f'/kaggle/working/dan_lambda_{lam}.pth')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
                break

    model.load_state_dict(best_state)
    dan_sweep_models[lam] = model

    target_acc, target_f1 = report_domains(model, f"DAN (lambda={lam})")
    sep = domain_separability(model, f"DAN(lambda={lam})")
    dan_sweep_results[lam] = {
        'source_f1': best_mean_f1, 'target_acc': target_acc,
        'target_f1': target_f1, 'separability': sep
    }

print("\n--- DAN Controlled Design Study: lambda_mmd sweep ---")
print(f"{'Lambda':<10}{'Source F1':>12}{'Sketch Acc':>12}{'Sketch F1':>12}{'Separability':>14}")
for lam in [0.1, 1, 10]:
    r = dan_sweep_results[lam]

    print(f"{lam:<10}{r['source_f1']:>11.2f}%{r['target_acc']:>11.2f}%{r['target_f1']:>11.2f}%{r['separability']:>13.2f}%")


=== Training DAN with lambda_mmd=0.1 ===
  Epoch 1/30 | Class Loss: 90.8787 | MMD Loss: 61.6100 | Mean Source Val F1: 86.04%
  Epoch 2/30 | Class Loss: 45.5558 | MMD Loss: 43.1112 | Mean Source Val F1: 90.79%
  Epoch 3/30 | Class Loss: 35.7024 | MMD Loss: 39.6520 | Mean Source Val F1: 93.14%
  Epoch 4/30 | Class Loss: 24.9763 | MMD Loss: 39.9805 | Mean Source Val F1: 93.35%
  Epoch 5/30 | Class Loss: 21.5424 | MMD Loss: 35.7124 | Mean Source Val F1: 92.48%
  Epoch 6/30 | Class Loss: 18.2691 | MMD Loss: 34.8428 | Mean Source Val F1: 93.43%
  Epoch 7/30 | Class Loss: 17.2648 | MMD Loss: 35.5247 | Mean Source Val F1: 91.24%
  Epoch 8/30 | Class Loss: 14.4132 | MMD Loss: 33.3410 | Mean Source Val F1: 93.42%
  Epoch 9/30 | Class Loss: 11.4320 | MMD Loss: 32.6140 | Mean Source Val F1: 91.56%
  Epoch 10/30 | Class Loss: 11.9035 | MMD Loss: 31.9632 | Mean Source Val F1: 92.94%
  Epoch 11/30 | Class Loss: 12.3255 | MMD Loss: 31.7637 | Mean Source Val F1: 95.09%
  Epoch 12/30 | Class Loss: 8.00

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 9. Summary table
Quick sanity check across methods before moving on to the domain-separability
diagnostic and per-class analysis.


In [28]:
print(f"{'Method':<10}{'Sketch Acc':>12}{'Sketch F1':>12}{'Domain Sep.':>14}")
print(f"{'ERM':<10}{erm_target_acc:>11.2f}%{erm_target_f1:>11.2f}%{erm_sep:>13.2f}%")
print(f"{'DAN':<10}{dan_target_acc:>11.2f}%{dan_target_f1:>11.2f}%{dan_sep:>13.2f}%")
print(f"{'DANN':<10}{dann_target_acc:>11.2f}%{dann_target_f1:>11.2f}%{dann_sep:>13.2f}%")
print(f"{'CDAN':<10}{cdan_target_acc:>11.2f}%{cdan_target_f1:>11.2f}%{cdan_sep:>13.2f}%")

Method      Sketch Acc   Sketch F1   Domain Sep.
ERM             72.46%      74.41%        99.73%
DAN             73.58%      71.76%        92.58%
DANN            30.54%      14.34%       100.00%
CDAN            35.71%      37.05%        98.35%
